# ML-02 — Research Question and Provisional Lane

This notebook frames our ML research question and provisional lane selection for the FlyRank capstone project.

> Skill loaded: `framing-ml-problems` + `flyrank-data`

## 1. My lane (or freestyle) and why

**Selected Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**Why this lane:**
Managing organic content inventories across tens of thousands of pages requires allocating limited human editorial capacity to the highest-leverage pages. FlyRank clients operate large content portfolios where manual page-by-page audits are infeasible. By building a decision-support opportunity scoring system, we can rank pages that exhibit high search demand alongside signs of decay or staleness. This lane focuses directly on the core operational challenge: transforming raw search and engagement signals into a prioritized, evidence-backed review queue for content teams.

## 2. The question: decision, action, cost of a wrong call

### The Research Question
*Which content items (pages) should content editors and SEO strategists prioritize for refresh review to prevent or mitigate organic search traffic decline?*

- **Unit of Analysis:** A pseudonymized content item (`content_id`) for a given pseudonymized client (`client_id`) over a trailing 90-day observation window.
- **The Decision:** How to allocate a team's monthly editorial capacity (e.g., selecting the top 20–50 candidate pages per month out of thousands of inventory items) for content refresh, expansion, or re-optimization.
- **Who Acts & The Action:** A content editor or SEO specialist inspects the top-ranked candidate pages alongside transparent reason codes (e.g., `stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`) and takes specific operational action: updating outdated facts, expanding thin sections, re-aligning content intent, or pruning irrelevant content.
- **Cost of a Wrong Recommendation:**
  - **False Positive (recommending a healthy page):** Wastes 2–5 hours of skilled editorial time per page rewriting content that did not need changes, and risks destabilizing existing search rankings.
  - **False Negative (missing a decaying page):** Allows high-demand pages to erode silently, resulting in compounding loss of organic search visibility, traffic, and client revenue.
- **Why Data / ML Helps:**
  Search performance involves high-dimensional interactions between impression volume, position tiers, CTR, engagement rates, and content age. While simple heuristic rules capture isolated conditions, machine learning models combine multi-signal patterns to accurately rank candidates by expected decay risk. On the starter dataset, a learned Random Forest baseline achieves Precision@50 of **0.740** compared to **0.240** for simple rule combinations.

## 3. Quick look at the data (2-3 real numbers)

We load the anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`) to examine real inventory metrics that justify focusing on content refresh opportunity scoring.

In [1]:
import os
import pandas as pd
import numpy as np

# Locate the starter dataset robustly
possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv'
]

csv_path = None
for p in possible_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError("Starter dataset content_refresh_anonymized.csv not found.")

df = pd.read_csv(csv_path)

# 1. Total inventory and client scope
n_rows = len(df)
n_clients = df['client_id'].nunique()

# 2. Decline prevalence in inventory
declining_count = (df['trend_direction'] == 'down').sum()
declining_pct = (declining_count / n_rows) * 100

# 3. High-demand stale content (impressions >= 500 & days_since_last_update >= 180)
stale_demand_mask = (df['impressions_90d'] >= 500) & (df['days_since_last_update'] >= 180)
stale_demand_count = stale_demand_mask.sum()
stale_demand_pct = (stale_demand_count / n_rows) * 100

print(f"1. Dataset Scope: {n_rows:,} content items across {n_clients} distinct clients.")
print(f"2. Traffic Decay Prevalence: {declining_count:,} pages ({declining_pct:.2f}%) are currently in decline (trend_direction == 'down').")
print(f"3. Stale High-Demand Candidates: {stale_demand_count:,} pages ({stale_demand_pct:.2f}%) have >= 500 impressions but haven't been updated in >= 180 days.")

1. Dataset Scope: 30,000 content items across 32 distinct clients.
2. Traffic Decay Prevalence: 16,262 pages (54.21%) are currently in decline (trend_direction == 'down').
3. Stale High-Demand Candidates: 17 pages (0.06%) have >= 500 impressions but haven't been updated in >= 180 days.


In [2]:
# Additional signal insight: High impression pages sitting on Page 1/2 (avg position <= 20) with weak CTR (< 0.5%)
high_imp_low_ctr = (df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)
print(f"4. High-Impression Low-CTR Candidates: {high_imp_low_ctr.sum():,} pages sit on Page 1/2 (avg position 1-20) with CTR < 0.5%.")

4. High-Impression Low-CTR Candidates: 9,759 pages sit on Page 1/2 (avg position 1-20) with CTR < 0.5%.


## 4. Careful words: what I can and can't claim

### What We CAN Claim
- **Observed Metrics & Correlations:** We can quantify historical patterns between search signals (impressions, position tiers, CTR, freshness) and observed traffic outcomes.
- **Decision-Support Prioritization:** We can produce a ranked queue of candidate pages sorted by predicted decay risk or opportunity score to optimize editorial review workflows.
- **Model vs Baseline Comparisons:** We can demonstrate whether a machine learning ranking model achieves higher Precision@K than transparent rule baselines on holdout client evaluation sets.

### What We CANNOT Claim
- **Causal Proof of Recovery:** We cannot claim that refreshing a page guarantees a recovery in traffic or ranking. Proving causality requires controlled experiments or causal inference designs not present in observational snapshots.
- **Reverse-Engineering Google's Algorithm:** We cannot claim to have identified Google's internal ranking factors or SERP mechanics.
- **AI Citation or Generative Search Rankings:** We cannot claim visibility in LLM answers or AI citations; `ai_sessions_90d` measures only GA4 click-through traffic originating from AI referral sources.
- **Absolute Future Predictions:** We cannot claim deterministic future traffic predictions, as external market shifts, competitor updates, and search engine algorithm changes remain unobserved.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.